# Análisis de Seguridad con CodeQL


Este notebook ejecuta un análisis de seguridad estático sobre los repositorios clonados en `data/repos/` utilizando CodeQL, la herramienta de análisis de código de GitHub.

## ¿Qué es CodeQL?

CodeQL es un motor de búsqueda de código que permite escribir consultas para encontrar vulnerabilidades, problemas de seguridad y defectos de codificación en el código fuente. Produce resultados en formato SARIF (Static Analysis Results Interchange Format).

## Flujo de Ejecución

1. **Descubrimiento de repositorios**: Escanea `data/repos/` para encontrar todos los subdirectorios
2. **Creación de base de datos CodeQL**: Para cada repositorio, CodeQL crea una base de datos indexada del código
3. **Análisis**: Se ejecutan consultas de seguridad predefinidas en la base de datos
4. **Generación de resultados**: Los hallazgos se convierten a formato JSON normalizado y se guardan en `data/results/`

## Preparación: Importar `scripts/generate_codeql.py`

A continuación importaremos el módulo `CodeQLAnalyzer` desde el archivo `scripts/generate_codeql.py`.

In [4]:
import sys
from pathlib import Path
import json
import pandas as pd

# Agregar el directorio scripts al path para importar el modulo generate_codeql
# Este módulo contiene la clase CodeQLAnalyzer que orquesta todo el análisis
project_root = Path().cwd().parent.parent
scripts_path = project_root / "scripts"
if str(scripts_path) not in sys.path:
    sys.path.insert(0, str(scripts_path))

print(f"Notebook working directory: {Path().cwd()}")
print(f"Project root: {project_root}")
print(f"Scripts path: {scripts_path}")
print(f"Importing from: generate_codeql.py")

Notebook working directory: /workspaces/SBOMS-FASTAPI-2/nbs/vuln
Project root: /workspaces/SBOMS-FASTAPI-2
Scripts path: /workspaces/SBOMS-FASTAPI-2/scripts
Importing from: generate_codeql.py


In [5]:
# Importar la clase CodeQLAnalyzer desde generate_codeql.py
from generate_codeql import CodeQLAnalyzer

print("✓ Importado correctamente: CodeQLAnalyzer desde scripts/generate_codeql.py")

✓ Importado correctamente: CodeQLAnalyzer desde scripts/generate_codeql.py


## Configuración

In [6]:
# Configurar rutas
repos_path = project_root / "data" / "repos"
output_path = project_root / "data" / "results"

# Crear analizador
analizador = CodeQLAnalyzer(
    repos_path=str(repos_path),
    output_path=str(output_path)
)

# Configurar para ejecución normal (no dry-run)
analizador.dry_run = False

print(f"Repos path: {repos_path}")
print(f"Output path: {output_path}")
print(f"Dry run: {analizador.dry_run}")

Repos path: /workspaces/SBOMS-FASTAPI-2/data/repos
Output path: /workspaces/SBOMS-FASTAPI-2/data/results
Dry run: False


## Ejecución: Análisis CodeQL (usando `scripts/generate_codeql.py`)

Ahora ejecutaremos el análisis CodeQL sobre todos los repositorios descubiertos. El script `generate_codeql.py` manejará automáticamente la orquestación completa.

In [7]:
# Descubrir repositorios usando el método del CodeQLAnalyzer
# Este método viene de scripts/generate_codeql.py
repositorios = analizador.discover_repositories()
print(f"Repositorios encontrados: {len(repositorios)}")
for repo in repositorios:
    print(f"  - {repo}")

Repositorios encontrados: 9
  - data/repos/annotated-doc
  - data/repos/asyncer
  - data/repos/fastapi
  - data/repos/fastapi-cli
  - data/repos/fastapi-new
  - data/repos/fastapi-vscode
  - data/repos/full-stack-fastapi-template
  - data/repos/sqlmodel
  - data/repos/typer


In [5]:
# | eval: false
print("Iniciando análisis CodeQL con scripts/generate_codeql.py...")
analizador.run()
print("✓ Análisis completado")

INFO | Usando CodeQL CLI: /usr/local/bin/codeql
INFO | === Diagnóstico del Entorno CodeQL ===


Iniciando análisis CodeQL con scripts/generate_codeql.py...


INFO | ✓ CodeQL CLI: CodeQL command-line toolchain release 2.25.1.
INFO | ✓ Node.js: v20.20.2
INFO | ✓ npm: 10.8.2
INFO | Verificando query packs...
INFO | ✓ Query pack codeql/python-queries disponible
INFO | ✓ Query pack codeql/javascript-queries disponible
INFO | ✓ Query pack codeql/java-queries disponible
INFO | === Fin Diagnóstico ===

INFO | [1/9] Procesando repositorio data/repos/annotated-doc
INFO | Lenguaje detectado en annotated-doc: python
INFO | Directorio temporal CodeQL: /tmp/codeql_analysis
INFO | Creando base de datos CodeQL para annotated-doc (lenguaje: python)...
INFO | Usando suite compilada: /home/vscode/.codeql/packages/codeql/python-queries/1.8.0/codeql-suites/python-security-and-quality.qls
INFO | SARIF guardado en: /workspaces/SBOMS-FASTAPI-2/data/results/annotated-doc_temp.sarif (132805 bytes)
INFO | SARIF output length: 132805 bytes
INFO | parse_sarif: recibiendo 132805 bytes
INFO | parse_sarif: 1 runs encontrados
INFO | parse_sarif: 1 resultados en run[0]
INFO

✓ Análisis completado


## Inspección de Resultados

In [8]:
# | eval: false
# Listar archivos de resultados generados
output_path = project_root / "data" / "results"
codeql_files = sorted(output_path.glob("*-codeql.json"))

print(f"📁 Buscando en: {output_path}")
print(f"✓ Archivos de análisis CodeQL encontrados: {len(codeql_files)}")
for archivo in codeql_files:
    tamaño = archivo.stat().st_size / 1024  # KB
    print(f"  - {archivo.name} ({tamaño:.1f} KB)")

📁 Buscando en: /workspaces/SBOMS-FASTAPI-2/data/results
✓ Archivos de análisis CodeQL encontrados: 9
  - annotated-doc-codeql.json (0.8 KB)
  - asyncer-codeql.json (1.1 KB)
  - fastapi-cli-codeql.json (1.1 KB)
  - fastapi-codeql.json (35.5 KB)
  - fastapi-new-codeql.json (0.4 KB)
  - fastapi-vscode-codeql.json (2.7 KB)
  - full-stack-fastapi-template-codeql.json (12.1 KB)
  - sqlmodel-codeql.json (38.4 KB)
  - typer-codeql.json (43.3 KB)


In [ ]:
# | eval: false
# Cargar y mostrar resumen de UN análisis
results_dir = project_root / "data" / "results"
results_files = sorted(results_dir.glob("*-codeql.json"))

print(f"📂 Buscando en: {results_dir}")
print(f"📋 Archivos encontrados: {len(results_files)}")

if results_files:
    # Se modifica para leer todos los repositorios encontrados, no solo el primero
    for result_file in results_files:
        print(f"📖 Leyendo: {result_file.name}")

        with open(result_file, 'r', encoding='utf-8') as f:
            data_content = json.load(f)

        repo_name_clean = result_file.stem.replace('-codeql', '')
        total_found = data_content.get('total_issues', 0)

        print(f"\n📊 ANÁLISIS: {repo_name_clean}")
        print(f"{'='*50}")
        print(f"✓ Total de problemas encontrados: {total_found}")

        if total_found > 0:
            severity_data = data_content.get('issues_by_severity', {})
            print(f"\n🔍 Problemas por severidad:")
            for severity_type, count in severity_data.items():
                if count > 0:
                    emoji = "🔴" if severity_type == "error" else "🟡" if severity_type == "warning" else "⚪"
                    print(f"  {emoji} {severity_type}: {count}")
        else:
            print("⚠️  No se encontraron problemas")
else:
    print("❌ No se encontraron archivos de análisis CodeQL")

📂 Buscando en: /workspaces/SBOMS-FASTAPI-2/data/results
📋 Archivos encontrados: 9
📖 Leyendo: annotated-doc-codeql.json

📊 ANÁLISIS: annotated-doc
✓ Total de problemas encontrados: 1

🔍 Problemas por severidad:
  🟡 warning: 1
📖 Leyendo: asyncer-codeql.json

📊 ANÁLISIS: asyncer
✓ Total de problemas encontrados: 2

🔍 Problemas por severidad:
  🟡 warning: 2
📖 Leyendo: fastapi-cli-codeql.json

📊 ANÁLISIS: fastapi-cli
✓ Total de problemas encontrados: 2

🔍 Problemas por severidad:
  🟡 warning: 2
📖 Leyendo: fastapi-codeql.json

📊 ANÁLISIS: fastapi
✓ Total de problemas encontrados: 88

🔍 Problemas por severidad:
  🟡 warning: 88
📖 Leyendo: fastapi-new-codeql.json

📊 ANÁLISIS: fastapi-new
✓ Total de problemas encontrados: 0
⚠️  No se encontraron problemas
📖 Leyendo: fastapi-vscode-codeql.json

📊 ANÁLISIS: fastapi-vscode
✓ Total de problemas encontrados: 6

🔍 Problemas por severidad:
  🟡 warning: 6
📖 Leyendo: full-stack-fastapi-template-codeql.json

📊 ANÁLISIS: full-stack-fastapi-template
✓ Total

In [11]:
# | eval: false
# Mostrar problemas detallados
output_path_det = project_root / "data" / "results"
codeql_files_det = sorted(output_path_det.glob("*-codeql.json"))

if codeql_files_det:
    archivo_det = codeql_files_det[0]

    with open(archivo_det, 'r', encoding='utf-8') as f:
        analisis_det = json.load(f)

    issues_list = analisis_det.get('issues', [])

    print(f"📋 Leyendo: {archivo_det.name}")
    print(f"📊 Total de issues en archivo: {len(issues_list)}")

    if issues_list:
        # Crear DataFrame con los problemas
        df_issues = pd.DataFrame(issues_list)

        # Seleccionar y reordenar columnas
        cols_mostrar = ['rule_id', 'level', 'message', 'file']
        cols_disponibles = [c for c in cols_mostrar if c in df_issues.columns]
        df_mostrar = df_issues[cols_disponibles].head(20)

        print(f"\n📋 Primeros 20 problemas encontrados:")
        print(f"{'='*100}")
        for idx, row in df_mostrar.iterrows():
            print(f"\n{idx+1}. [{row['level'].upper()}] {row['rule_id']}")
            print(f"   📝 {row['message'][:80]}")
            if 'file' in row and pd.notna(row['file']):
                print(f"   📂 {row['file']}")

        if len(issues_list) > 20:
            print(f"\n... y {len(issues_list)-20} problemas más")
    else:
        print("⚠️  No hay detalles de problemas en el archivo")

📋 Leyendo: annotated-doc-codeql.json
📊 Total de issues en archivo: 1

📋 Primeros 20 problemas encontrados:

1. [WARNING] py/comparison-of-identical-expressions
   📝 Comparison of identical values; use cmath.isnan() if testing for not-a-number.
   📂 tests/test_main.py


## Análisis Detallado de Problemas

In [12]:
# | eval: false
# Compilar estadísticas consolidadas
from collections import defaultdict

output_path_stats = project_root / "data" / "results"
codeql_files_stats = sorted(output_path_stats.glob("*-codeql.json"))

stats_consolidadas = {
    'total_repositories': len(codeql_files_stats),
    'total_issues': 0,
    'issues_by_repo': {},
    'issues_by_severity': defaultdict(int),
    'top_rules': defaultdict(int)
}

for archivo in codeql_files_stats:
    with open(archivo, 'r', encoding='utf-8') as f:
        data = json.load(f)

    repo_name = archivo.stem.replace('-codeql', '')
    total = data.get('total_issues', 0)

    stats_consolidadas['total_issues'] += total
    stats_consolidadas['issues_by_repo'][repo_name] = total

    # Contar por severidad
    for severity, count in data.get('issues_by_severity', {}).items():
        stats_consolidadas['issues_by_severity'][severity] += count

    # Contar top rules
    for issue in data.get('issues', []):
        rule = issue.get('rule_id', 'unknown')
        stats_consolidadas['top_rules'][rule] += 1

# Convertir defaultdicts a dicts para visualización
stats_consolidadas['issues_by_severity'] = dict(
    stats_consolidadas['issues_by_severity'])
stats_consolidadas['top_rules'] = dict(sorted(
    stats_consolidadas['top_rules'].items(),
    key=lambda x: x[1],
    reverse=True
)[:10])

print("\n📊 ESTADÍSTICAS CONSOLIDADAS DE CODEQL")
print(f"{'='*60}")
print(
    f"\n📁 Repositorios analizados: {stats_consolidadas['total_repositories']}")
print(f"🔴 Total de problemas: {stats_consolidadas['total_issues']}")

print(f"\n📋 Problemas por repositorio:")
for repo, count in stats_consolidadas['issues_by_repo'].items():
    print(f"  • {repo}: {count}")

print(f"\n⚠️  Problemas por severidad:")
for severity, count in sorted(stats_consolidadas['issues_by_severity'].items()):
    emoji = "🔴" if severity == "error" else "🟡" if severity == "warning" else "⚪"
    print(f"  {emoji} {severity}: {count}")

print(f"\n🎯 Top 10 reglas más comunes:")
for rule, count in list(stats_consolidadas['top_rules'].items())[:10]:
    print(f"  • {rule}: {count}")


📊 ESTADÍSTICAS CONSOLIDADAS DE CODEQL

📁 Repositorios analizados: 9
🔴 Total de problemas: 317

📋 Problemas por repositorio:
  • annotated-doc: 1
  • asyncer: 2
  • fastapi-cli: 2
  • fastapi: 88
  • fastapi-new: 0
  • fastapi-vscode: 6
  • full-stack-fastapi-template: 27
  • sqlmodel: 98
  • typer: 93

⚠️  Problemas por severidad:
  🔴 error: 0
  ⚪ note: 0
  🟡 warning: 317

🎯 Top 10 reglas más comunes:
  • py/unsafe-cyclic-import: 71
  • py/ineffectual-statement: 57
  • py/unused-import: 40
  • py/unused-local-variable: 27
  • js/insecure-randomness: 26
  • py/mixed-returns: 14
  • py/cyclic-import: 13
  • py/empty-except: 9
  • py/non-iterable-in-for-loop: 8
  • py/not-named-self: 7


## Próximos Pasos

Los resultados del análisis CodeQL se han guardado en `data/results/` con el patrón `{repo-name}-codeql.json`.

### Cómo usar estos resultados:

1. **Revisar vulnerabilidades críticas**: Filtrar por `level: "error"` para identificar problemas de seguridad críticos
2. **Seguimiento**: Integrar con sistemas de seguimiento de mermas (issues, PRs) para gestionar remediación
3. **Análisis de tendencias**: Comparar resultados entre repositorios y períodos de tiempo
4. **Automatizar**: Integrar estos análisis en CI/CD para detectar problemas continuamente

#### Para agregar nuevos repositorios:

1. **Edita `data/repos.json`** y agrega nuevos repositorios:

```json
{
    "repositories": [
        {
            "url": "https://github.com/owner/repo-name.git",
            "path": "data/repos/repo-name",
            "ref": "v1.0.0"
        }
    ]
}
```

2. **Ejecuta en la terminal** (esto clonará los repositorios y los agregará como submódulos):

```bash
uv run scripts/add_submodules.py && git submodule update --init --recursive
```

3. **Ejecuta el análisis** nuevamente desde el notebook o la terminal:

```bash
uv run scripts/generate_codeql.py
```